`FakeQuantizer` rounds weights and activations onto a narrower grid and leaves the result in floating point. The model keeps its `float32` tensors and its ordinary modules; while it is rounded it also holds a floating-point copy of every weight it touched, so it occupies more memory than it did before and pays for the rounding work on every forward pass. The one thing it measures is what a width costs in **accuracy**.

This page rounds three fine-tuned classifiers — `vgg16_bn`, `resnet18` and `efficientnet_b0` — to the same grid of widths and reports what each one costs on Imagenette. Training, calibration and validation all go through fastai: `vision_learner`, `fine_tune`, and `learn.validate()` behind every number below.

In [ ]:
#| include: false
import warnings
warnings.filterwarnings("ignore")

## Setup

In [ ]:
import torch, torch.nn as nn
from fastai.vision.all import *
from fasterai.quantize.fake_quantizer import FakeQuantizer
from fasterai.core.precision import fake_quant_spec

Imagenette at 160 pixels, one `fine_tune(1)` per architecture — one frozen epoch and one unfrozen one. That is enough to give each model a floating-point accuracy to be measured against; it is not a training recipe.

In [ ]:
path = untar_data(URLs.IMAGENETTE_160)
dls = ImageDataLoaders.from_folder(path, valid='val', item_tfms=Resize(160), bs=32)
N = len(dls.valid_ds)
print(f'{N} validation images, {len(dls.train_ds)} training images')

3925 validation images, 9469 training images


In [ ]:
def finetune(arch):
    "A fastai `vision_learner` fine-tuned for one cycle on Imagenette"
    set_seed(42, reproducible=True)
    learn = vision_learner(dls, arch, metrics=accuracy)
    with learn.no_bar(), learn.no_logging():
        learn.fine_tune(1)
    return learn

def score(learn):
    "Validation accuracy of `learn`, as a percentage and as k/n"
    with learn.no_bar():
        acc = float(learn.validate()[1])
    return 100 * acc, round(acc * N), N

learners = {'vgg16_bn': finetune(vgg16_bn),
            'resnet18': finetune(resnet18),
            'efficientnet_b0': finetune(efficientnet_b0)}

for name, learn in learners.items():
    pct, k, n = score(learn)
    print(f'{name:<18}{pct:.2f}% ({k}/{n})')

vgg16_bn          96.92% (3804/3925)
resnet18          96.00% (3768/3925)
efficientnet_b0   95.67% (3755/3925)


Those three are the references every row below is read against, all on the same 3925 validation images.

## What rounding does not do

The model does not get smaller and it does not get faster. `quantize_model()` writes rounded values back into the same `float32` tensors and registers a buffer holding the original of every weight it rounds, so the model grows while it is rounded and shrinks back on `remove()`.

In [ ]:
def nbytes(m):
    "Bytes held by every parameter and buffer of `m`"
    return (sum(p.numel() * p.element_size() for p in m.parameters())
            + sum(b.numel() * b.element_size() for b in m.buffers()))

model = learners['resnet18'].model
before = {k: v.detach().clone() for k, v in model.state_dict().items()}

fq = FakeQuantizer(model, weight_bits=8, act_bits=8, qscheme='per_channel')
fq.calibrate(dls.train, n_batches=5)
fq.quantize_model()

print(f'dtype while rounded  {next(model.parameters()).dtype}')
print(f'bytes while rounded  {nbytes(model):,}')
print(f'spec while rounded   {fake_quant_spec(model).label}')

fq.remove()
print(f'bytes after remove() {nbytes(model):,}')
print(f'spec after remove()  {fake_quant_spec(model)}')
print(f'weights restored     {all(torch.equal(before[k], v) for k, v in model.state_dict().items())}')

dtype while rounded  torch.float32
bytes while rounded  93,672,288
spec while rounded   W8A8
bytes after remove() 46,886,832
spec after remove()  None
weights restored     True


Rounded, this ResNet-18 holds 93,672,288 bytes of parameters and buffers against 46,886,832 before. `remove()` puts the original weights back byte for byte and drops the spec.

Four more things stay outside the rounding:

- **Biases stay in floating point.** An integer kernel would carry them at int32; this class does not model that.
- **BatchNorm is not folded** into the convolution it follows. Folding changes the weights that get rounded, so a folded model and an unfolded one give different answers at the same width — [`BN_Folder`](../../misc/bn_folding.html) does the folding.
- **`nn.Embedding` and `nn.ConvTranspose2d` are outside the default `layer_type`**, which is `(nn.Conv2d, nn.Linear)`. A model holding them keeps those tensors byte-identical while the report still names a width.
- **A rounded model is not a QAT model.** Its weights are baked: they sit on the grid now, and training would walk them straight off it. Quantization-aware training is not in this version.

`print_precision()` names the width every layer carries. Here `layer_bits` holds the first convolution in floating point:

In [ ]:
fq = FakeQuantizer(model, weight_bits=8, act_bits=8, qscheme='per_channel',
                   layer_bits={'0.0': None})
fq.calibrate(dls.train, n_batches=5)
fq.quantize_model()
fq.print_precision()
fq.remove()


Simulated Precision Report:
--------------------------------------------------------------------------------
Layer                            Type           Weight     Act        Weight axis 
--------------------------------------------------------------------------------
0.0                              Conv2d         float      8 bits     -           
0.4.0.conv1                      Conv2d         8 bits     8 bits     per_channel 
0.4.0.conv2                      Conv2d         8 bits     8 bits     per_channel 
0.4.1.conv1                      Conv2d         8 bits     8 bits     per_channel 
0.4.1.conv2                      Conv2d         8 bits     8 bits     per_channel 
0.5.0.conv1                      Conv2d         8 bits     8 bits     per_channel 
0.5.0.conv2                      Conv2d         8 bits     8 bits     per_channel 
0.5.0.downsample.0               Conv2d         8 bits     8 bits     per_channel 
0.5.1.conv1                      Conv2d         8 bits     8 b

## The widths, three models

Each configuration is built on the fine-tuned model, validated, and removed before the next one starts. Activations at 8 bits are calibrated on five training batches with the default `observer='static'`; `act_bits=None` leaves them in floating point.

In [ ]:
def measure(learn, **kw):
    "Round `learn.model` to one configuration, validate it, and restore the floating-point weights"
    fq = FakeQuantizer(learn.model, **kw)
    try:
        if fq.observer == 'static' and fq.act_bits is not None:
            fq.calibrate(dls.train, n_batches=5)
        fq.quantize_model()
        return score(learn)
    finally:
        fq.remove()

CONFIGS = {
    'W8A8 static, per_channel':     dict(weight_bits=8, act_bits=8,    qscheme='per_channel'),
    'W8 weights-only, per_channel': dict(weight_bits=8, act_bits=None, qscheme='per_channel'),
    'W8 weights-only, per_tensor':  dict(weight_bits=8, act_bits=None, qscheme='per_tensor'),
    'W4A8 static, per_channel':     dict(weight_bits=4, act_bits=8,    qscheme='per_channel'),
    'W4 weights-only, per_channel': dict(weight_bits=4, act_bits=None, qscheme='per_channel'),
    'W2A8 static, per_channel':     dict(weight_bits=2, act_bits=8,    qscheme='per_channel'),
    'W2 weights-only, per_channel': dict(weight_bits=2, act_bits=None, qscheme='per_channel'),
}

rows = {'floating point': {n: score(l) for n, l in learners.items()}}
for label, kw in CONFIGS.items():
    rows[label] = {n: measure(l, **kw) for n, l in learners.items()}

print(f'{"":<32}' + ''.join(f'{name:<19}' for name in learners))
for label, per_arch in rows.items():
    cells = ''.join(f'{pct:.2f}% {k}/{n}'.ljust(19) for pct, k, n in
                    (per_arch[name] for name in learners))
    print(f'{label:<32}{cells}')

                                vgg16_bn           resnet18           efficientnet_b0    
floating point                  96.92% 3804/3925   96.00% 3768/3925   95.67% 3755/3925   
W8A8 static, per_channel        96.97% 3806/3925   95.87% 3763/3925   44.08% 1730/3925   
W8 weights-only, per_channel    96.94% 3805/3925   96.05% 3770/3925   95.69% 3756/3925   
W8 weights-only, per_tensor     96.97% 3806/3925   96.03% 3769/3925   95.77% 3759/3925   
W4A8 static, per_channel        94.60% 3713/3925   91.46% 3590/3925   18.11% 711/3925    
W4 weights-only, per_channel    94.68% 3716/3925   91.59% 3595/3925   43.11% 1692/3925   
W2A8 static, per_channel        10.42% 409/3925    10.06% 395/3925    9.99% 392/3925     
W2 weights-only, per_channel    10.42% 409/3925    10.06% 395/3925    9.63% 378/3925     


The three models do not agree.

**At 8 bits on the weights alone**, all three land within a few images of their floating-point reference: 96.94% (3805/3925) against 96.92% (3804/3925) for `vgg16_bn`, 96.05% (3770/3925) against 96.00% (3768/3925) for `resnet18`, and 95.69% (3756/3925) against 95.67% (3755/3925) for `efficientnet_b0`. Two of the three rounded rows come out a hair above their reference, which is what a difference of one or two images looks like.

**Adding 8-bit activations splits them.** `vgg16_bn` and `resnet18` stay where they were, at 96.97% (3806/3925) and 95.87% (3763/3925), while `efficientnet_b0` falls to 44.08% (1730/3925). Its weights survive 8 bits; its activations do not.

**At 4 bits** the order changes again: `vgg16_bn` gives up about two points, 94.68% (3716/3925) against 96.92% (3804/3925); `resnet18` gives up more, 91.59% (3595/3925) against 96.00% (3768/3925); and `efficientnet_b0` is down at 43.11% (1692/3925).

**At 2 bits** all three have stopped separating the classes: 10.42% (409/3925), 10.06% (395/3925) and 9.63% (378/3925) over ten classes.

## Where the scale comes from

`qscheme` picks the axis the scale is fitted on. `per_tensor` fits one scale to the whole weight tensor; `per_channel` fits one per output row.

At 8 bits the two axes land within a few images of each other on all three models, and they do not order consistently: `per_tensor` comes out above `per_channel` on `vgg16_bn`, 96.97% (3806/3925) against 96.94% (3805/3925), and on `efficientnet_b0`, 95.77% (3759/3925) against 95.69% (3756/3925) — and below it on `resnet18`, 96.03% (3769/3925) against 96.05% (3770/3925). Differences of one to three images out of 3925 do not establish that either axis dominates.

What the axis does change reliably is which weights survive at all. One scale has to cover the loudest row of the tensor, so the quiet rows round away:

In [ ]:
def dead_filters(learn, **kw):
    "Output filters a rounding turns entirely to zero"
    fq = FakeQuantizer(learn.model, **kw)
    try:
        fq.quantize_model()
        return sum(int((m.weight.flatten(1).abs().sum(1) == 0).sum())
                   for m in learn.model.modules() if isinstance(m, (nn.Conv2d, nn.Linear)))
    finally:
        fq.remove()

for name, learn in learners.items():
    per_tensor = dead_filters(learn, weight_bits=4, act_bits=None, qscheme='per_tensor')
    per_channel = dead_filters(learn, weight_bits=4, act_bits=None, qscheme='per_channel')
    print(f'{name:<18}per_tensor {per_tensor:5d}   per_channel {per_channel:5d}')

vgg16_bn          per_tensor    82   per_channel     6
resnet18          per_tensor    15   per_channel     4
efficientnet_b0   per_tensor   413   per_channel     0


At 4 bits, `per_tensor` empties 82 output filters of `vgg16_bn`, 15 of `resnet18` and 413 of `efficientnet_b0`; `per_channel` empties 6, 4 and 0. That is what the axis decides — not a uniformly higher accuracy, but a different failure.

## Groups, and a refusal

`qscheme='per_group'` cuts each row into fixed blocks that share one scale, which means the block size has to divide the row. A convolution row is `in_channels` times the kernel height times the kernel width, and a ResNet-18 stem gives an odd one. `FakeQuantizer` says so before it writes anything:

In [ ]:
try:
    FakeQuantizer(learners['resnet18'].model, weight_bits=4, act_bits=None,
                  qscheme='per_group', group_size=64)
except ValueError as e:
    print(e)

group_size=64 does not divide the 147 weights of a row of '0.0': pass a group_size that divides it, or qscheme='per_channel'.


Narrowing `layer_type` to the layers whose rows do divide is one way through — here, the two `nn.Linear` layers of the fastai head:

In [ ]:
for name, learn in learners.items():
    pct, k, n = measure(learn, weight_bits=4, act_bits=None, qscheme='per_group',
                        group_size=64, layer_type=nn.Linear)
    print(f'{name:<18}{pct:.2f}% ({k}/{n})')

vgg16_bn          97.02% (3808/3925)
resnet18          95.92% (3765/3925)
efficientnet_b0   95.80% (3760/3925)


With only the head rounded to 4 bits in groups, the three models sit within a handful of images of their floating-point references: 97.02% (3808/3925), 95.92% (3765/3925) and 95.80% (3760/3925). The row shows that the axis works on the tensors whose shape admits it, not that 4 bits is cheap — almost nothing was rounded.

## One layer held at floating point

`layer_bits` maps a layer name to its own width, and `None` holds that layer in floating point. The first convolution is a common one to hold out: it carries the fewest weights and every later layer reads through it.

In [ ]:
first_conv = {name: next(n for n, m in learn.model.named_modules()
                         if isinstance(m, nn.Conv2d))
              for name, learn in learners.items()}
print(first_conv)

for name, learn in learners.items():
    pct, k, n = measure(learn, weight_bits=4, act_bits=None, qscheme='per_channel',
                        layer_bits={first_conv[name]: None})
    print(f'{name:<18}{pct:.2f}% ({k}/{n})')

{'vgg16_bn': '0.0.0', 'resnet18': '0.0', 'efficientnet_b0': '0.0.0.0'}
vgg16_bn          95.57% (3751/3925)
resnet18          92.48% (3630/3925)
efficientnet_b0   45.17% (1773/3925)


Holding the first convolution out of the 4-bit rounding moves `vgg16_bn` from 94.68% (3716/3925) to 95.57% (3751/3925), `resnet18` from 91.59% (3595/3925) to 92.48% (3630/3925), and `efficientnet_b0` from 43.11% (1692/3925) to 45.17% (1773/3925). It buys back part of what 4 bits cost on the two convolutional stacks, and leaves `efficientnet_b0` far from its 95.67% (3755/3925) reference.

## Where EfficientNet loses it

The W8A8 row said the activations are what `efficientnet_b0` cannot carry at 8 bits. A forward pass over five training batches shows where their ranges are widest:

In [ ]:
eff = learners['efficientnet_b0']
ranges = {}

def watch(name):
    def hook(_, inp, out):
        lo, hi = out.detach().amin().item(), out.detach().amax().item()
        if name in ranges: lo, hi = min(ranges[name][0], lo), max(ranges[name][1], hi)
        ranges[name] = (lo, hi)
    return hook

handles = [m.register_forward_hook(watch(name)) for name, m in eff.model.named_modules()
           if isinstance(m, (nn.Conv2d, nn.Linear))]
eff.model.eval()
with torch.no_grad():
    for i, batch in enumerate(dls.train):
        if i >= 5: break
        eff.model(batch[0])
for h in handles: h.remove()

widest = sorted(ranges.items(), key=lambda kv: kv[1][1] - kv[1][0], reverse=True)
for name, (lo, hi) in widest[:5]:
    print(f'{name:<22}[{lo:8.1f}, {hi:8.1f}]')

0.0.4.0.block.0.0     [  -518.4,    136.2]
0.0.2.0.block.0.0     [  -228.4,    149.3]
0.0.3.0.block.0.0     [  -190.4,    119.4]
0.0.1.0.block.0.0     [  -178.5,     90.3]
0.0.2.1.block.0.0     [  -167.4,     89.1]


All five are the pointwise expansion at the head of an inverted-residual block, and the widest runs from -518.4 to 136.2. One 8-bit grid stretched over that span leaves little resolution for the values near zero, which is most of them.

`layer_act_bits` holds named sites at floating point the way `layer_bits` does for weights:

In [ ]:
for k in (0, 5, 20):
    pct, kk, n = measure(eff, weight_bits=8, act_bits=8, qscheme='per_channel',
                         layer_act_bits={name: None for name, _ in widest[:k]} or None)
    print(f'{k:2d} sites in floating point: {pct:.2f}% ({kk}/{n})')

 0 sites in floating point: 45.89% (1801/3925)
 5 sites in floating point: 72.79% (2857/3925)
20 sites in floating point: 77.94% (3059/3925)


Leaving the five widest sites in floating point lifts the model from 45.89% (1801/3925) to 72.79% (2857/3925), and twenty sites reach 77.94% (3059/3925) — still well short of the 95.67% (3755/3925) it started from. The widest sites are where the damage concentrates, and holding them out does not undo it.

The 45.89% (1801/3925) on the first line is the same configuration as the 44.08% (1730/3925) in the grid above. The two differ because `observer='static'` freezes its scales on whichever five batches `calibrate` draws from the shuffled training loader, and the two calls drew different ones. Every A8 number on this page is one such draw.

---

## Summary

| Call | What it does |
|---|---|
| `FakeQuantizer(model, weight_bits=8, act_bits=8)` | a quantizer bound to this model; `None` on either width leaves those tensors in floating point |
| `fq.calibrate(dls.train, n_batches=5)` | observes activation ranges and freezes the static scales; takes a fastai `DataLoaders` or one of its loaders |
| `fq.quantize_model()` | rounds the weights in place and hooks the activations |
| `fq.remove()` | restores the floating-point weights and drops every hook and buffer |
| `fq.print_precision()` | the per-layer width report |
| `qscheme='per_tensor'` / `'per_channel'` / `'per_group'` | one scale for the tensor, one per output row, or one per block of a row |
| `group_size=64` | the block size `per_group` shares a scale over; it has to divide the row |
| `layer_type=nn.Linear` | narrows the modules that carry the rounding; the default is `(nn.Conv2d, nn.Linear)` |
| `layer_bits={name: None}` | one weight width per layer, `None` holding it at floating point |
| `layer_act_bits={name: None}` | the same, for that layer's activations |
| `observer='dynamic'` | recomputes activation scales per batch instead of freezing them; not exercised on this page |

Every accuracy on this page is one `learn.validate()` pass, and the page measures accuracy only — rounding in floating point changes neither the size of the model nor, in any direction worth reporting, its speed.

> **Measured on:** Imagenette-160, 9469 training images and 3925 validation images, at 160 pixels with batch size 32. One `fine_tune(1)` per architecture under `set_seed(42, reproducible=True)`, then one `learn.validate()` per configuration on the same 3925 images. Activation scales come from five training batches drawn from the shuffled loader, so the A8 rows move by a few images from call to call. Single run throughout: no repetition, no dispersion, no confidence interval. The device is not recorded, because nothing here is a latency.

## See Also

- [FakeQuantizer API](../../quantize/fake_quantizer.html) - the class, its arguments and its refusals
- [Precision grammar](../../core/precision.html) - the widths and axes fasterai names, and which backends run them
- [Quantizer](../../quantize/quantizer.html) - the backends that produce a model that is actually smaller
- [Quantization Methods Compared](quantization_compared.html) - size and latency of real quantized models
- [Deployable Export](deployable_export.html) - exporting a quantized model and checking what came out
- [BN Folding](../misc/bn_folding.html) - folding batch norm into the convolution ahead of it